In [3]:
"""Task D: Training and Evaluation Pipeline"""

import torch
import torch.nn as nn
from torch.optim import Adam
import matplotlib.pyplot as plt
import time
import os
from tqdm import tqdm

from dataset import get_dataloaders
from models.baseline import GRUForecast
from models.igru_variant import iGRUForecast
from utils import set_seed, EarlyStopping, calculate_all_metrics

class Trainer:
    """Complete training pipeline for time series forecasting"""
    
    def __init__(self, model, model_name, device='cuda'):
        self.model = model.to(device)
        self.model_name = model_name
        self.device = device
        self.criterion = nn.MSELoss()
        
    def train_epoch(self, loader, optimizer):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        num_batches = 0
        
        for x, y in tqdm(loader, desc=f"Training {self.model_name}", leave=False):
            x, y = x.to(self.device), y.to(self.device)
            
            optimizer.zero_grad()
            y_pred = self.model(x)
            loss = self.criterion(y_pred, y)
            loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        return total_loss / num_batches
    
    def validate_epoch(self, loader):
        """Validate for one epoch"""
        self.model.eval()
        total_loss = 0
        num_batches = 0
        
        with torch.no_grad():
            for x, y in tqdm(loader, desc=f"Validating {self.model_name}", leave=False):
                x, y = x.to(self.device), y.to(self.device)
                y_pred = self.model(x)
                loss = self.criterion(y_pred, y)
                total_loss += loss.item()
                num_batches += 1
        
        return total_loss / num_batches
    
    def train(self, train_loader, val_loader, epochs=20, lr=0.001, 
              patience=7, save_best=True):
        """Full training loop with early stopping"""
        
        optimizer = Adam(self.model.parameters(), lr=lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=3, factor=0.5
        )
        early_stopping = EarlyStopping(patience=patience)
        
        train_losses = []
        val_losses = []
        best_val_loss = float('inf')
        
        print(f"\n{'='*60}")
        print(f"Training {self.model_name}")
        print(f"{'='*60}")
        print(f"Parameters: {sum(p.numel() for p in self.model.parameters()):,}")
        print(f"Device: {self.device}")
        print(f"Learning rate: {lr}")
        print(f"Epochs: {epochs}")
        print(f"{'='*60}\n")
        
        start_time = time.time()
        
        for epoch in range(epochs):
            # Train and validate
            train_loss = self.train_epoch(train_loader, optimizer)
            val_loss = self.validate_epoch(val_loader)
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            
            # Learning rate scheduling
            scheduler.step(val_loss)
            
            # Save best model
            if val_loss < best_val_loss and save_best:
                best_val_loss = val_loss
                torch.save(self.model.state_dict(), f"results/best_{self.model_name}.pt")
            
            # Early stopping
            if early_stopping(val_loss):
                print(f"Early stopping at epoch {epoch+1}")
                break
            
            # Print progress
            print(f"\nEpoch {epoch+1}/{epochs}")
            print(f"  Train Loss: {train_loss:.6f}")
            print(f"  Val Loss:   {val_loss:.6f}")
            print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        training_time = time.time() - start_time
        
        # Load best model
        if save_best:
            self.model.load_state_dict(torch.load(f"results/best_{self.model_name}.pt"))
        
        return train_losses, val_losses, training_time
    
    def evaluate(self, test_loader):
        """Evaluate model on test set"""
        self.model.eval()
        
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for x, y in tqdm(test_loader, desc="Testing", leave=False):
                x, y = x.to(self.device), y.to(self.device)
                y_pred = self.model(x)
                all_preds.append(y_pred.cpu())
                all_targets.append(y.cpu())
        
        y_pred = torch.cat(all_preds, dim=0)
        y_true = torch.cat(all_targets, dim=0)
        
        metrics = calculate_all_metrics(y_pred, y_true)
        
        return metrics, y_pred, y_true

def plot_training_curves(train_losses, val_losses, model_name):
    """Plot training and validation loss curves"""
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss', linewidth=2)
    plt.plot(val_losses, label='Validation Loss', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.title(f'Training Curves - {model_name}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f'results/loss_curve_{model_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

def plot_predictions(y_true, y_pred, model_name, horizon=24, num_samples=3):
    """Plot sample predictions vs ground truth"""
    fig, axes = plt.subplots(num_samples, 1, figsize=(12, 3*num_samples))
    if num_samples == 1:
        axes = [axes]
    
    for i in range(num_samples):
        axes[i].plot(y_true[i, :, 0].numpy(), label='Ground Truth', linewidth=2)
        axes[i].plot(y_pred[i, :, 0].numpy(), label='Prediction', linewidth=2, linestyle='--')
        axes[i].set_title(f'Sample {i+1} - {model_name}')
        axes[i].set_xlabel('Forecast Horizon (hours)')
        axes[i].set_ylabel('Normalized Value')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'results/predictions_{model_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

def main():
    """Main execution function"""
    
    # Setup
    set_seed(42)
    os.makedirs("results", exist_ok=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load data
    print("\nLoading data...")
    train_loader, val_loader, test_loader, scaler, input_dim = get_dataloaders(batch_size=64)
    print(f"Input dimension: {input_dim}")
    
    # Model configurations
    model_configs = {
        'GRU': GRUForecast(input_dim, hidden_dim=64, horizon=24, out_dim=1, num_layers=2),
        'iGRU': iGRUForecast(input_dim, hidden_dim=64, horizon=24, out_dim=1, num_layers=2)
    }
    
    results = {}
    
    # Train and evaluate each model
    for model_name, model in model_configs.items():
        print(f"\n{'='*70}")
        print(f"Model: {model_name}")
        print(f"{'='*70}")
        
        trainer = Trainer(model, model_name, device)
        
        # Train
        train_losses, val_losses, training_time = trainer.train(
            train_loader, val_loader, epochs=20, lr=0.001, patience=7
        )
        
        # Plot training curves
        plot_training_curves(train_losses, val_losses, model_name)
        
        # Evaluate
        metrics, y_pred, y_true = trainer.evaluate(test_loader)
        metrics['Training Time (s)'] = training_time
        metrics['Parameters'] = sum(p.numel() for p in model.parameters())
        
        # Plot predictions
        plot_predictions(y_true, y_pred, model_name)
        
        results[model_name] = metrics
        print(f"\n{model_name} Results:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")
    
    # Comparison table
    print("\n" + "="*70)
    print("FINAL COMPARISON")
    print("="*70)
    print(f"{'Model':<10} {'RMSE':<10} {'MAE':<10} {'MAPE':<10} {'Params':<12} {'Time(s)':<10}")
    print("-" * 62)
    for name, metrics in results.items():
        print(f"{name:<10} {metrics['RMSE']:<10.4f} {metrics['MAE']:<10.4f} "
              f"{metrics['MAPE']:<10.2f} {metrics['Parameters']:<12,} {metrics['Training Time (s)']:<10.1f}")
    
    return results

if __name__ == "__main__":
    results = main()

ModuleNotFoundError: No module named 'dataset'